# 17.1 Environments — `venv`, `pip` and Dependencies

**Prerequisites:** 07 Module and Packages, 15 Testing and Debugging  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 Why a global install is a trap, and what isolation actually buys
- `venv` — what it creates, and what “activating” really does
- 🔴 `pip` vs `python -m pip` — the wrong-interpreter bug
- `sys.path` and `site-packages`: where imports actually come from
- Declared dependencies vs `pip freeze` — and why they are not the same file
- Version specifiers, and pinning that does not hurt
- Transitive dependencies and conflicts
- The modern landscape: `uv`, `poetry`, `pdm`, `pipx`

---

## The problem isolation solves

You have two projects. One needs `requests` 2.28, the other needs 2.34. Install both globally
and one of them breaks — there is exactly one `site-packages` per interpreter, and one version
of each package in it.

```
   WITHOUT isolation                    WITH a venv per project
   ┌────────────────────────┐           ┌──────────┐  ┌──────────┐
   │  system site-packages  │           │ .venv A  │  │ .venv B  │
   │  requests 2.34 (only)  │           │ req 2.28 │  │ req 2.34 │
   └────────────────────────┘           └──────────┘  └──────────┘
     project A: broken                    both work, and neither can
     project B: fine                      break the system Python
```

There is a second, quieter reason. On Linux and macOS the system Python is **used by the
operating system**. Installing into it — or worse, upgrading something in it — can break system
tools. Modern distributions now refuse outright with an
`externally-managed-environment` error, which is that lesson turned into a guard rail.

> **The rule:** one virtual environment per project, created in the project directory, never
> committed to git.

In [ ]:
import shutil
import subprocess
import sys
import sysconfig
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py171_"))


def write(rel, source, root=None):
    path = (root or WORK) / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return path


def run(args, cwd=None, timeout=600, label=None):
    """Run a command and return its output, with the command echoed."""
    done = subprocess.run(args, cwd=cwd or WORK, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=timeout)
    shown = label or " ".join(
        "python" if a == sys.executable else str(a) for a in args)
    body = (done.stdout + done.stderr).strip() or "(no output)"
    return (f"$ {shown}\n" + "-" * 68 + "\n" + body
            + "\n" + "-" * 68 + f"\nexit code: {done.returncode}")


print("scratch:", WORK)

## What `venv` actually creates

```bash
python -m venv .venv
```

That is the whole command. What it produces is worth looking at, because "activating an
environment" sounds like magic and is not.

In [ ]:
ENV = WORK / "demo-venv"

print(run([sys.executable, "-m", "venv", str(ENV)], timeout=300,
          label=f"python -m venv {ENV.name}"))

print()
print("what it created:")
for path in sorted(ENV.iterdir()):
    kind = "dir " if path.is_dir() else "file"
    print(f"   {kind}  {path.name}")

bindir = ENV / ("Scripts" if sys.platform == "win32" else "bin")
print(f"\ninside {bindir.name}/ :")
for path in sorted(bindir.iterdir())[:10]:
    print("   ", path.name)

print("\npyvenv.cfg - the whole configuration:")
print(textwrap.indent((ENV / "pyvenv.cfg").read_text(encoding="utf-8").strip(), "   "))

Three things in there matter:

| Item | What it is |
|---|---|
| `pyvenv.cfg` | 🔴 the entire mechanism — it names the **base interpreter** and whether system packages are visible |
| `Scripts/` (or `bin/`) | a `python` executable and the console scripts of installed packages |
| `Lib/site-packages/` | where this environment's packages live |

### 🔴 "Activating" is just `PATH`

`activate` prepends `Scripts/`(or `bin/`) to your `PATH` and sets `VIRTUAL_ENV`. That is all it
does. It does **not** change what an already-running program sees, and it is not required — you
can always call the environment's interpreter by full path:

```bash
.venv/Scripts/python -m pytest      # works with no activation at all
```

That form is what CI scripts and this curriculum's own harness use, because it cannot be
ambiguous.

In [ ]:
venv_python = bindir / ("python.exe" if sys.platform == "win32" else "python")

print(run([str(venv_python), "-c",
           "import sys; print('executable :', sys.executable);"
           "print('prefix     :', sys.prefix);"
           "print('base_prefix:', sys.base_prefix);"
           "print('in a venv  :', sys.prefix != sys.base_prefix)"],
          label="demo-venv/python -c ..."))

print()
print("...and the interpreter running this notebook:")
print(f"   executable : {sys.executable}")
print(f"   prefix     : {sys.prefix}")
print(f"   in a venv  : {sys.prefix != sys.base_prefix}")

🔴 **`sys.prefix != sys.base_prefix` is how you detect a virtual environment**
in code. `sys.base_prefix` points at the interpreter the venv was created *from*; inside a venv
they differ.

## 🔴 `pip` vs `python -m pip`

This is the single most common environment bug, and the cause of a thousand
*"but I installed it!"* messages.

Typing `pip` runs **whichever `pip` is first on your `PATH`**. That may belong to a completely
different interpreter than the `python` you are about to run. The package installs successfully
— into the wrong environment.

```
   $ pip install requests          -> installs into  /usr/bin/python3
   $ python my_script.py           -> runs           ~/proj/.venv/bin/python
   ImportError: No module named 'requests'
```

**`python -m pip` cannot have this problem**: it installs into the interpreter you just named.

In [ ]:
print("which pip would run:", shutil.which("pip") or "(none on PATH)")
print("which python would run:", shutil.which("python") or "(none on PATH)")
print("the interpreter running THIS cell:", sys.executable)
print()
print("🔴 If the first two lines point somewhere other than the third,")
print("   `pip install X` puts X where this interpreter cannot see it.")
print()
print("The unambiguous form always agrees with itself:")
print(run([sys.executable, "-m", "pip", "--version"], label="python -m pip --version"))

## Where imports actually come from

`import x` searches `sys.path` **in order** and stops at the first match (**07**). Knowing the
order explains most import surprises.

In [ ]:
import site

print("sys.path, in search order:")
for i, entry in enumerate(sys.path):
    label = entry or "(the current directory)"
    print(f"   {i}: {label}")

print("\nsite-packages for this environment:")
for path in site.getsitepackages():
    print("   ", path)

print("\npurelib (where pip installs):", sysconfig.get_path("purelib"))

print()
print("🔴 Entry 0 is searched FIRST. A file called random.py or json.py in your")
print("   working directory will shadow the standard library module of that name.")

> **The shadowing trap.** A file named `random.py`, `json.py`, `email.py` or `types.py` in
> your project directory wins over the standard library. The symptom is a bizarre
> `AttributeError` from a module you did not write. **07** covers imports properly; the fix is
> always to rename your file.

## Declaring dependencies

Two files, often confused, doing different jobs:

| | What it holds | Who writes it |
|---|---|---|
| **declared** dependencies | what your project *needs* — `requests>=2.28` | 🔴 **you**, by hand |
| **lock** / `pip freeze` output | every package and exact version installed | a tool, generated |

`pip freeze > requirements.txt` produces the second and people commit it as the first. That
gives you a file listing forty packages, of which you directly use four — and no record of
which four.

Modern practice puts declared dependencies in **`pyproject.toml`** (**17.2**) and lets a lock
file be generated. `requirements.txt` remains everywhere, so it is worth reading fluently.

In [ ]:
listing = run([sys.executable, "-m", "pip", "list", "--format=columns"],
              label="python -m pip list").splitlines()
for line in listing[:16]:
    print(line)
print(f"   ... and {max(0, len(listing) - 19)} more packages")

In [ ]:
frozen = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True, encoding="utf-8").stdout.splitlines()

print(f"pip freeze produced {len(frozen)} lines. The first eight:")
for line in frozen[:8]:
    print("   ", line)

print()
print("🔴 Of those, this project DIRECTLY uses only a handful -")
print("   pytest, mypy, ruff, hypothesis, coverage. The rest are")
print("   transitive: dependencies of dependencies.")
print()
print("A declared list says what you asked for:")
print(textwrap.indent(textwrap.dedent('''
    pytest>=8
    mypy>=1.8
    ruff
''').strip(), "   "))

## Version specifiers

| Specifier | Means | Use when |
|---|---|---|
| `requests` | any version | 🔴 almost never — a major release will break you |
| `requests>=2.28` | at least | a library, where you must not over-constrain |
| `requests>=2.28,<3` | a range | **the usual choice for a library** |
| `requests~=2.28.0` | compatible release — `>=2.28.0,<2.29.0` | you rely on patch-level behaviour |
| `requests==2.34.2` | exactly | 🔴 **an application**, in a lock file |

🔴 **The rule that resolves most arguments:**

> **Libraries declare ranges. Applications pin exactly.**

A library that pins `requests==2.34.2` is unusable alongside anything else that needs a
different version. An application that does not pin gets a different set of dependencies on
every deploy — and a bug you cannot reproduce (**15.9**).

## Transitive dependencies

You install one package; you get twelve. Each has its own requirements, and pip must find a set
that satisfies all of them at once.

In [ ]:
print(run([sys.executable, "-m", "pip", "show", "build"],
          label="python -m pip show build"))
print()
shown = run([sys.executable, "-m", "pip", "show", "hypothesis"],
            label="python -m pip show hypothesis").splitlines()
for line in shown:
    if line.startswith(("$", "-", "Name", "Version", "Requires",
                        "Required-by", "exit")):
        print(line)

`Requires:` and `Required-by:` are the edges of the dependency graph
(**14.9**). When two packages need incompatible versions of a third, you get a **resolution
conflict** — pip will tell you, at length.

Three ways out, in order:

1. **Loosen your own pins.** Often you pinned tighter than you needed.
2. **Upgrade the older package.** The conflict usually means one is behind.
3. **Vendor or isolate.** Last resort — separate services, or `pipx` for tools.

> 🔴 **`pipx` is the answer for command-line tools.** `ruff`, `black`, `httpie` and friends are
> *applications*, not libraries — they should not share an environment with your project at all.
> `pipx install ruff` gives each its own hidden venv with a single command on your `PATH`.

## Installing without touching your environment

`pip install --target DIR` puts a package in a directory of your choosing. It is how you inspect
a package, and how this notebook demonstrates installation without modifying the environment
running it.

In [ ]:
TARGET = WORK / "isolated"

installed = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
     "--target", str(TARGET), "pytest"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=600)

if installed.returncode != 0:
    print("$ python -m pip install --no-deps --target isolated/ pytest")
    print("   this step needs network access (or a warm pip cache); skipping.")
    print("   " + installed.stderr.strip().splitlines()[-1][:90])
else:
    print("$ python -m pip install --no-deps --target isolated/ pytest   -> ok")

print()
print("what landed in the target directory:")
for path in sorted(TARGET.iterdir())[:8] if TARGET.exists() else []:
    print("   ", path.name)

if TARGET.exists():
    print()
    print(run([sys.executable, "-c",
               f"import sys; sys.path.insert(0, r'{TARGET}');"
               "import pytest; print('imported pytest', pytest.__version__);"
               f"print('loaded from the target dir:', "
               f"pytest.__file__.startswith(r'{TARGET}'))"],
              label="python -c 'import pytest from the target dir'"))

The package was importable purely because its directory was on `sys.path`
— which is all `site-packages` ever was.

## The modern landscape

`venv` + `pip` is the standard-library baseline and always works. Several tools build on it:

| Tool | What it is | Why you would use it |
|---|---|---|
| **`venv` + `pip`** | stdlib | 🔴 always available; no install needed |
| **`uv`** | a very fast installer and resolver in Rust | 10–100× faster; drop-in `pip`/`venv` replacement |
| **`poetry`** | dependency management + packaging + publishing | one tool for the whole lifecycle |
| **`pdm`** | similar, standards-first | PEP-621 native |
| **`pipx`** | installs *applications* in isolated venvs | CLI tools, not libraries |
| **`conda`** | environments including non-Python deps | scientific stacks, compilers, CUDA |

They all read the same `pyproject.toml` (**17.2**) for project metadata, so the choice is less
consequential than it looks. 🔴 **Learn `venv` and `pip` first** — every tool above is explained
in terms of them, and every CI system already has them.

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **`pip install` instead of `python -m pip install`.** The package goes to whichever pip is first on `PATH`, which may not be the interpreter you then run.
2. **Installing into the system Python.** On Linux and macOS it belongs to the OS; modern distributions now block it with `externally-managed-environment`.
3. 🔴 **Committing `pip freeze` output as your declared dependencies.** It lists forty packages when you asked for four, and records no intent.
4. **Committing the `.venv` directory.** It is large, machine-specific, and rebuilt in one command. Add it to `.gitignore`.
5. **Pinning exact versions in a library.** It makes your library uninstallable alongside anything else.
6. **Not pinning in an application.** Every deploy gets a different dependency set, and the bug you cannot reproduce is one of them (**15.9**).
7. **Naming a file after a standard-library module** — `random.py`, `json.py`, `types.py`. The current directory is searched first and shadows it.
8. **Installing CLI tools into your project venv.** `ruff` and friends are applications; use `pipx`.
9. **Assuming `activate` is required.** It only edits `PATH`; calling `.venv/Scripts/python` directly is unambiguous and works everywhere.

## Best Practices

- One venv per project, in the project directory, named `.venv`, and gitignored.
- Always `python -m pip`, never bare `pip`.
- Declare dependencies by hand in `pyproject.toml` (**17.2**); let lock files be generated.
- Ranges for libraries (`>=2.28,<3`), exact pins for applications.
- Keep development dependencies separate — an `optional-dependencies` group or a `requirements-dev.txt`.
- Use `pipx` for anything you invoke as a command rather than import.
- Record the Python version your project needs (`requires-python`) so mistakes surface at install time.
- Recreate your environment from scratch occasionally — it is the only way to know your declared dependencies are complete.

## Practice Exercises

Try these before moving on.

1. Create a venv, activate it, and confirm `sys.prefix != sys.base_prefix`. Then deactivate and run the same check by full path instead.
2. 🔴 Find out which `pip` and which `python` your shell resolves. Do they belong to the same environment? Prove it with `python -m pip --version`.
3. Create `json.py` in a directory, then `import json` from that directory and explain the error. Which `sys.path` entry caused it?
4. Run `pip freeze` in an environment of your own. How many packages did you actually ask for, and how many are transitive?
5. Write a `requirements.txt` using four different specifier styles and explain when each is right.
6. 🔴 Delete a project's venv and rebuild it from your declared dependencies alone. Did anything fail to import? That is a missing declaration.
7. Use `pip install --target` to install a package into a directory, then import it by manipulating `sys.path`. What does that tell you about what `site-packages` is?
8. **Interview question:** a colleague says “it works on my machine but not in CI”. Name three environment causes and how you would check each.

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | `venv` no longer installs `setuptools` by default — packages must declare their build requirements |
| **3.11** | `venv` gained `--upgrade-deps`; `tomllib` added for reading `pyproject.toml` (**17.2**) |
| **3.11+** | `externally-managed-environment` (PEP 668) — the OS Python refuses `pip install` |
| **3.9** | `python -m venv` became the clear standard; `virtualenv` remains for older versions and extra features |

## Where next

| Notebook | Covers |
|---|---|
| **17.2** | `pyproject.toml` — dependencies, metadata and every tool's configuration in one file |
| **17.3** | building and publishing a package |
| **17.4** | linting and formatting with `ruff` |
| **17.5** | profiling and performance |

The folder index and a one-sentence summary are at the end of **17.5**.

## Related

- **07 Module and Packages** — imports, `sys.path` and packages
- **15.6 Testing in Practice** — why `src/` layout and `pip install -e .` matter for tests
- **15.9** — "works on my machine" as a reproducibility problem
- **16.6** — the same ratchet idea, applied to type strictness